In [58]:
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
import platform
import torch
import numpy as np
import pandas as pd
import re
from collections import defaultdict
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from typing import Any, Dict, List, Optional, Union
import torch.nn as nn

In [12]:
raw_labels = """T001,Organism
T002,Plant
T004,Fungus
T005,Virus
T007,Bacterium
T008,Animal
T010,Vertebrate
T011,Amphibian
T012,Bird
T013,Fish
T014,Reptile
T015,Mammal
T016,Human
T017,Anatomical Structure
T018,Embryonic Structure
T019,Congenital Abnormality
T020,Acquired Abnormality
T021,Fully Formed Anatomical Structure
T022,Body System
T023,"Body Part, Organ, or Organ Component"
T024,Tissue
T025,Cell
T026,Cell Component
T028,Gene or Genome
T029,Body Location or Region
T030,Body Space or Junction
T031,Body Substance
T032,Organism Attribute
T033,Finding
T034,Laboratory or Test Result
T037,Injury or Poisoning
T038,Biologic Function
T039,Physiologic Function
T040,Organism Function
T041,Mental Process
T042,Organ or Tissue Function
T043,Cell Function
T044,Molecular Function
T045,Genetic Function
T046,Pathologic Function
T047,Disease or Syndrome
T048,Mental or Behavioral Dysfunction
T049,Cell or Molecular Dysfunction
T050,Experimental Model of Disease
T051,Event
T052,Activity
T053,Behavior
T054,Social Behavior
T055,Individual Behavior
T056,Daily or Recreational Activity
T057,Occupational Activity
T058,Health Care Activity
T059,Laboratory Procedure
T060,Diagnostic Procedure
T061,Therapeutic or Preventive Procedure
T062,Research Activity
T063,Molecular Biology Research Technique
T064,Governmental or Regulatory Activity
T065,Educational Activity
T066,Machine Activity
T067,Phenomenon or Process
T068,Human-caused Phenomenon or Process
T069,Environmental Effect of Humans
T070,Natural Phenomenon or Process
T071,Entity
T072,Physical Object
T073,Manufactured Object
T074,Medical Device
T075,Research Device
T077,Conceptual Entity
T078,Idea or Concept
T079,Temporal Concept
T080,Qualitative Concept
T081,Quantitative Concept
T082,Spatial Concept
T083,Geographic Area
T085,Molecular Sequence
T086,Nucleotide Sequence
T087,Amino Acid Sequence
T088,Carbohydrate Sequence
T089,Regulation or Law
T090,Occupation or Discipline
T091,Biomedical Occupation or Discipline
T092,Organization
T093,Health Care Related Organization
T094,Professional Society
T095,Self-help or Relief Organization
T096,Group
T097,Professional or Occupational Group
T098,Population Group
T099,Family Group
T100,Age Group
T101,Patient or Disabled Group
T102,Group Attribute
T103,Chemical
T104,Chemical Viewed Structurally
T109,Organic Chemical
T114,"Nucleic Acid, Nucleoside, or Nucleotide"
T116,"Amino Acid, Peptide, or Protein"
T120,Chemical Viewed Functionally
T121,Pharmacologic Substance
T122,Biomedical or Dental Material
T123,Biologically Active Substance
T125,Hormone
T126,Enzyme
T127,Vitamin
T129,Immunologic Factor
T130,"Indicator, Reagent, or Diagnostic Aid"
T131,Hazardous or Poisonous Substance
T167,Substance
T168,Food
T169,Functional Concept
T170,Intellectual Product
T171,Language
T184,Sign or Symptom
T185,Classification
T190,Anatomical Abnormality
T191,Neoplastic Process
T192,Receptor
T194,Archaeon
T195,Antibiotic
T196,"Element, Ion, or Isotope"
T197,Inorganic Chemical
T200,Clinical Drug
T201,Clinical Attribute
T203,Drug Delivery Device
T204,Eukaryote"""

In [22]:
def parse_multi_report_pubtator(raw_data):
    # If the input is bytes, decode it to a string first
    if isinstance(raw_data, bytes):
        raw_data = raw_data.decode('utf-8')

    report_groups = defaultdict(lambda: {'title': '', 'abstract': '', 'annotations': []})
    lines = raw_data.strip().split('\n')
    
    # 1. Grouping Phase: Separate data by PMID
    for line in lines:
        if not line.strip(): 
            continue
        
        parts = line.split('|')
        if len(parts) > 2:
            pmid, type_flag, content = parts[0], parts[1], parts[2]
            if type_flag == 't':
                report_groups[pmid]['title'] = content
            elif type_flag == 'a':
                report_groups[pmid]['abstract'] = content
        else:
            tab_parts = line.split('\t')
            if len(tab_parts) >= 5:
                pmid = tab_parts[0]
                report_groups[pmid]['annotations'].append({
                    'start': int(tab_parts[1]),
                    'end': int(tab_parts[2]),
                    'tags': tab_parts[4].split(',')
                })

    all_results = []

    # 2. Processing Phase: Handle each report independently
    for pmid, data in report_groups.items():
        # Using a space to separate title and abstract to maintain offset integrity
        full_text = data['title'] + " " + data['abstract']
        
        report_words = []
        report_tags = []
        
        for match in re.finditer(r'\w+', full_text):
            start, end = match.start(), match.end()
            word = match.group()
            
            current_word_tags = set()
            for ann in data['annotations']:
                if start >= ann['start'] and end <= ann['end']:
                    for t in ann['tags']:
                        current_word_tags.add(t)
            
            report_words.append(word)
            
            # --- Logic Update Start ---
            # If the set is empty, we tag it with 'O'
            if not current_word_tags:
                report_tags.append(['O'])
            else:
                report_tags.append(sorted(list(current_word_tags)))
            # --- Logic Update End ---
        
        all_results.append({
            'pmid': pmid,
            'words': report_words,
            'tags': report_tags
        })
        
    return all_results

In [23]:
@dataclass
class DataCollatorForMultiLabelTokenClassification:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        labels = [feature.pop("labels") for feature in features]
        
        # Standard padding for input_ids, attention_mask, etc.
        batch = self.tokenizer.pad(
            features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        # Manual padding for the multi-hot label vectors
        sequence_length = batch["input_ids"].shape[1]
        num_labels = len(labels[0][0])
        batch_size = len(labels)
        
        # Create a padded tensor of zeros for labels
        padded_labels = torch.zeros((batch_size, sequence_length, num_labels))
        
        for i, label_set in enumerate(labels):
            # Truncate if the labels are longer than the padded sequence
            end = min(len(label_set), sequence_length)
            padded_labels[i, :end, :] = torch.tensor(label_set[:end])
            
        batch["labels"] = padded_labels
        return batch

In [79]:
with open("metadata/corpus_pubtator_pmids_train.txt") as f:
    train_text = [line.strip() for line in f if line.strip()]
with open("metadata/corpus_pubtator_pmids_test.txt") as f:
    test_text = [line.strip() for line in f if line.strip()]
with open("metadata/corpus_pubtator_pmids_dev.txt") as f:
    dev_text = [line.strip() for line in f if line.strip()]

In [80]:
train_text

['27773526',
 '27583127',
 '27634004',
 '27987734',
 '28238559',
 '27505348',
 '27618756',
 '28436286',
 '27838489',
 '27438188',
 '27981855',
 '27681182',
 '27859264',
 '27555654',
 '27871915',
 '27496388',
 '27929135',
 '28405379',
 '27446232',
 '28282614',
 '28484250',
 '28388318',
 '27638646',
 '28337431',
 '27255399',
 '27958282',
 '27938026',
 '28548949',
 '27656565',
 '27737807',
 '28416098',
 '27906083',
 '27512378',
 '28217505',
 '28497418',
 '27853635',
 '28066530',
 '27861606',
 '28346915',
 '27776603',
 '28117911',
 '27405997',
 '27413416',
 '27583405',
 '27348352',
 '27681017',
 '28332360',
 '27653361',
 '28055203',
 '27684099',
 '27445319',
 '27362260',
 '28438945',
 '27848974',
 '27721784',
 '28542641',
 '27744641',
 '27420101',
 '27958356',
 '27465497',
 '28544860',
 '27306621',
 '27812347',
 '27371349',
 '27826671',
 '28349200',
 '27594346',
 '27683791',
 '28249118',
 '28284468',
 '27736701',
 '28194801',
 '27310824',
 '27612680',
 '27536134',
 '27716325',
 '28084988',

In [83]:
# Loading the full text
with open("metadata/corpus_pubtator.txt", "r") as f:
    full_text = f.read()

results = parse_multi_report_pubtator(full_text)
corpus_id_dict = {item['pmid']: item for item in results}
train_results = []
dev_results = []
test_results = []

for pmid in corpus_id_dict:
    if pmid in train_text:
        train_results.append(corpus_id_dict[pmid])
    if pmid in dev_text:
        dev_results.append(corpus_id_dict[pmid])
    else:
        test_results.append(corpus_id_dict[pmid])


results = results = results[:5]
results_predict = results[6:7]

In [40]:
def align_labels_with_tokens(tokenizer, words, tags, label_to_id):
    # Tokenize the full list of words
    tokenized_inputs = tokenizer(
        words, 
        is_split_into_words=True, 
        truncation=True, 
        max_length=512,
        padding='max_length'
    )
    
    word_ids = tokenized_inputs.word_ids()
    new_labels = []
    
    for word_idx in word_ids:
        if word_idx is None:
            # Padding/Special tokens get all zeros
            new_labels.append([0.0] * len(label_to_id)) 
        else:
            # This maps the multi-hot vector of the word to the specific sub-token
            multi_hot = [0.0] * len(label_to_id)
            current_tags = tags[word_idx]
            if current_tags != ['O']:
                for tag in current_tags:
                    if tag in label_to_id:
                        multi_hot[label_to_id[tag]] = 1.0
            new_labels.append(multi_hot)

    return tokenized_inputs, new_labels


class AlignPubTator(torch.utils.data.Dataset):
    def __init__(self, parsed_data, tokenizer, label2id):
        self.data = parsed_data
        self.tokenizer = tokenizer
        self.label2id = label2id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs, labels = align_labels_with_tokens(
            self.tokenizer, 
            item['words'], 
            item['tags'], 
            self.label2id
        )
        
        # Remove the batch dimension added by tokenizer() call inside the function
        # and ensure everything is a list for the data collator to handle
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": labels
        }

In [62]:
class MultiLabelTokenTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Flatten the logits and labels for BCEWithLogitsLoss
        # Logits shape: [Batch, Seq_Len, Num_Labels] -> [Batch * Seq_Len, Num_Labels]
        # Labels shape: [Batch, Seq_Len, Num_Labels] -> [Batch * Seq_Len, Num_Labels]
        active_loss = inputs["attention_mask"].view(-1) == 1
        
        active_logits = logits.view(-1, self.model.config.num_labels)
        active_labels = labels.view(-1, self.model.config.num_labels)
        
        # Only calculate loss on non-padded tokens
        # (This ensures padding doesn't skew your gradients)
        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(active_logits[active_loss], active_labels[active_loss])
        
        return (loss, outputs) if return_outputs else loss

In [84]:
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

label_list = [line.split(',')[0] for line in raw_labels.strip().split('\n')]
label2id =  {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForMultiLabelTokenClassification(tokenizer)

train_data = AlignPubTator(
    parsed_data=train_results, 
    tokenizer=tokenizer, 
    label2id=label2id
)
val_data = AlignPubTator(
    parsed_data = dev_results,
    tokenizer = tokenizer,
    label2id=label2id
)


test_dataset = AlignPubTator(
    parsed_data=results_predict,
    tokenizer=tokenizer,
    label2id=label2id
)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    label2id = label2id,
    id2label = id2label,

)
model.config.problem_type = "multi_label_classification"
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") #might have issues on macbook.
else:
    device = torch.device("cpu")
model.to(device)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 61777.79it/s]
BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [46]:
full_dataset

In [47]:
# Check the first item
sample = full_dataset[0]

#print("Input IDs shape:", sample['input_ids'].shape) # Should be [512]
print("Labels shape:", torch.tensor(sample['labels']).shape) # Should be [512, len(label_list)]

# Verify if any labels are actually '1' (not just an empty document)
labels_tensor = torch.tensor(sample['labels'])
if (labels_tensor == 1.0).any():
    print("Found positive labels in sample!")
else:
    print("Warning: Sample 0 has no annotations (all 'O').")

Labels shape: torch.Size([512, 127])
Found positive labels in sample!


In [85]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size = 16,
    num_train_epochs = 3,
)

trainer = MultiLabelTokenTrainer(
    model = model,
    args = training_args,
    train_dataset = train_data,
    eval_dataset=val_data,
    data_collator=data_collator #need training set
)

In [ ]:
trainer.train()

/opt/anaconda3/envs/NLP-bio/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG15XFamilyCommandBuffer: 0x455716aa0>
    label = <none> 
    device = <AGXG15SDevice: 0x11f53cc00>
        name = Apple M3 Pro 
    commandQueue = <AGXG15XFamilyCommandQueue: 0x177af6a00>
        label = <none> 
        device = <AGXG15SDevice: 0x11f53cc00>
            name = Apple M3 Pro 
    retainedReferences = 1
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG15XFamilyCommandBuffer: 0x4555b29a0>
    label = <none> 
    device = <AGXG15SDevice: 0x11f53cc00>
        name = Apple M3 Pro 
    commandQueue = <AGXG15XFamilyCommandQueue: 0x177af6a00>
        label = <none> 
        device =

In [73]:
test_data = """Pseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease and shorter survival, and chronic Pa infection (CPA) is associated with reduced lung function, faster rate of lung decline, increased rates of exacerbations and shorter survival. By using exome sequencing and extreme phenotype design, it was recently shown that isoforms of dynactin 4 (DCTN4) may influence Pa infection in CF, leading to worse respiratory disease. The purpose of this study was to investigate the role of DCTN4 missense variants on Pa infection incidence, age at first Pa infection and chronic Pa infection incidence in a cohort of adult CF patients from a single centre. Polymerase chain reaction and direct sequencing were used to screen DNA samples for DCTN4 variants. A total of 121 adult CF patients from the Cochin Hospital CF centre have been included, all of them carrying two CFTR defects: 103 developed at least 1 pulmonary infection with Pa, and 68 patients of them had CPA. DCTN4 variants were identified in 24% (29/121) CF patients with Pa infection and in only 17% (3/18) CF patients with no Pa infection. Of the patients with CPA, 29% (20/68) had DCTN4 missense variants vs 23% (8/35) in patients without CPA. Interestingly, p.Tyr263Cys tend to be more frequently observed in CF patients with CPA than in patients without CPA (4/68 vs 0/35), and DCTN4 missense variants tend to be more frequent in male CF patients with CPA bearing two class II mutations than in male CF patients without CPA bearing two class II mutations (P = 0.06). Our observations reinforce that DCTN4 missense variants, especially p.Tyr263Cys, may be involved in the pathogenesis of CPA in male CF.
25763772	0	5	DCTN4	T116,T123	C4308010"""

In [ ]:
NER_Pipeline = pipeline(
    "ner",
    model = model,
    tokenizer=tokenizer,
    device = device 
    )

In [74]:
def predict_biomed_ner(text, threshold=0.5):
    # 1. Tokenize and get word_ids
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    word_ids = inputs.word_ids(batch_index=0)
    
    # 2. Forward pass
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).squeeze(0) # Remove batch dim
    
    # 3. Process results
    results = []
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # We track word_idx to avoid repeating labels for sub-words if desired,
    # but in multi-label, we usually just want to see what's happening per token.
    for i, word_idx in enumerate(word_ids):
        if word_idx is None: continue # Skip [CLS], [SEP], [PAD]
        
        token_probs = probs[i]
        # Get indices where probability > threshold
        active_label_indices = (token_probs > threshold).nonzero(as_tuple=True)[0]
        
        if len(active_label_indices) > 0:
            active_labels = [id2label[idx.item()] for idx in active_label_indices]
            results.append({
                "token": tokens[i],
                "word_idx": word_idx,
                "labels": active_labels,
                "confidence": [round(token_probs[idx].item(), 4) for idx in active_label_indices]
            })
            
    return results
results_test = predict_biomed_ner(test_data)

In [71]:
results_test

[{'token': 'lipid',
  'word_idx': 0,
  'labels': ['T004',
   'T005',
   'T011',
   'T012',
   'T013',
   'T015',
   'T016',
   'T021',
   'T022',
   'T023',
   'T024',
   'T025',
   'T026',
   'T030',
   'T033',
   'T034',
   'T037',
   'T039',
   'T040',
   'T041',
   'T042',
   'T045',
   'T047',
   'T049',
   'T052',
   'T053',
   'T054',
   'T059',
   'T062',
   'T063',
   'T065',
   'T067',
   'T068',
   'T069',
   'T070',
   'T072',
   'T074',
   'T075',
   'T078',
   'T080',
   'T083',
   'T088',
   'T090',
   'T091',
   'T092',
   'T093',
   'T095',
   'T097',
   'T099',
   'T102',
   'T103',
   'T114',
   'T122',
   'T125',
   'T126',
   'T127',
   'T129',
   'T131',
   'T168',
   'T170',
   'T184',
   'T185',
   'T192',
   'T196',
   'T197',
   'T200',
   'T201',
   'T203'],
  'confidence': [0.5767,
   0.6101,
   0.5368,
   0.5135,
   0.5433,
   0.5577,
   0.5147,
   0.6839,
   0.5471,
   0.574,
   0.509,
   0.5233,
   0.5087,
   0.6491,
   0.5771,
   0.536,
   0.5032,
   0.5

In [75]:
def get_readable_predictions(text):
    raw_results = predict_biomed_ner(text)
    word_groups = defaultdict(set)
    
    # Group sub-tokens back into words
    for res in raw_results:
        for label in res['labels']:
            word_groups[res['word_idx']].add(label)
    
    # Map back to the original string
    words = text.split() # Simple split for display
    for word_idx, labels in word_groups.items():
        if word_idx < len(words):
            print(f"Word: {words[word_idx]} | Labels: {list(labels)}")
get_readable_predictions(test_data)

Word: Pseudomonas | Labels: ['T201', 'T053', 'T030', 'T200', 'T171', 'T087', 'T129', 'T090', 'T168', 'T052', 'T095', 'T039', 'T069', 'T023', 'T191', 'T196', 'T070', 'T131', 'T005', 'T080', 'T092', 'T103', 'T127', 'T007', 'T057', 'T078', 'T016', 'T170', 'T059', 'T101', 'T197', 'T075', 'T067', 'T064', 'T121', 'T040', 'T022', 'T060', 'T063', 'T013', 'T046', 'T012', 'T049', 'T072', 'T185', 'T203', 'T114', 'T015', 'T033', 'T025', 'T065', 'T068', 'T021', 'T085', 'T097', 'T098', 'T002', 'T083', 'T018', 'T041', 'T099', 'T062', 'T184', 'T011', 'T190', 'T195']
Word: aeruginosa | Labels: ['T201', 'T053', 'T030', 'T171', 'T087', 'T090', 'T168', 'T052', 'T095', 'T004', 'T069', 'T023', 'T191', 'T196', 'T070', 'T131', 'T005', 'T080', 'T092', 'T103', 'T034', 'T102', 'T007', 'T057', 'T078', 'T016', 'T170', 'T050', 'T059', 'T101', 'T197', 'T075', 'T067', 'T040', 'T022', 'T060', 'T031', 'T045', 'T063', 'T013', 'T049', 'T072', 'T185', 'T010', 'T203', 'T114', 'T015', 'T033', 'T025', 'T065', 'T068', 'T021',